# Validate expression split, round 4

This notebook validates the round-4 rule-based expression classifier in `data/classify_expressions.py`. The current classifier is a 3-way split: `attribute`, `positional`, and `relational`. Review an approximately balanced audit sample from the generated classification log, record whether each prediction is correct, and use the resulting judgments to estimate per-label precision for the README classifier-validation history.

For incorrect predictions, record the better label as one of `attribute`, `positional`, or `relational` so the audit remains useful for future error analysis.

In [ ]:
import os
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "configs" / "config.yaml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if not (repo_root / "configs" / "config.yaml").exists():
    raise RuntimeError("Could not locate repo root (configs/config.yaml not found in any parent directory).")
os.chdir(repo_root)
print("Working directory set to:", repo_root)

In [ ]:
from pathlib import Path

import pandas as pd

from common.utils import load_config

# Edit these before running the audit.
AUDIT_ROUND = 4
LOG_CSV_PATH = Path('data/splits/refcoco_val_classification_log.csv')
N_PER_LABEL = 100
RANDOM_SEED = 7
LABEL_ORDER = ['attribute', 'positional', 'relational']

df = pd.read_csv(LOG_CSV_PATH)
missing_columns = {'ref_id', 'expression', 'label', 'matched_cue'} - set(df.columns)
if missing_columns:
    raise ValueError(f"Classification log is missing expected columns: {sorted(missing_columns)}")

observed_labels = set(df['label'].dropna().unique())
unexpected_labels = observed_labels - set(LABEL_ORDER)
if unexpected_labels:
    raise ValueError(f"Unexpected classifier labels for round {AUDIT_ROUND}: {sorted(unexpected_labels)}")

labels_to_sample = [label for label in LABEL_ORDER if label in observed_labels]
sampled = pd.concat(
    [
        df[df['label'] == label].sample(n=min(N_PER_LABEL, (df['label'] == label).sum()), random_state=RANDOM_SEED)
        for label in labels_to_sample
    ],
    ignore_index=True,
)

log_stem = LOG_CSV_PATH.name.removesuffix('_classification_log.csv')
dataset, split = log_stem.rsplit('_', 1)
config = load_config('configs/config.yaml')
ground_truth = (
    pd.read_json(Path('data/processed') / f'{dataset}_{split}.jsonl', lines=True)
    [['ref_id', 'file_name', 'bbox_xyxy']]
    .drop_duplicates(subset='ref_id')
    .set_index('ref_id')
)
sampled = sampled.merge(ground_truth, how='left', left_on='ref_id', right_index=True)

label_counts = sampled['label'].value_counts().reindex(LABEL_ORDER, fill_value=0)
print(f"Round {AUDIT_ROUND} audit sample from {LOG_CSV_PATH}")
print(label_counts[label_counts > 0])
sampled.head()

In [ ]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

judgments = []
corrected_labels = []
review_notes = []

# Review one prediction at a time and record whether the predicted label is correct.
for row in sampled.itertuples(index=False):
    print()

    if pd.notna(row.file_name):
        image_path = Path(config['paths']['coco_images_dir']) / row.file_name
        try:
            with Image.open(image_path) as image:
                annotated_image = image.copy()
            ImageDraw.Draw(annotated_image).rectangle(row.bbox_xyxy, outline='red', width=3)
            plt.imshow(annotated_image)
            plt.axis('off')
            plt.show()
        except (FileNotFoundError, OSError):
            print(f"no image available for ref_id={row.ref_id}")
    else:
        print(f"no image available for ref_id={row.ref_id}")

    print(f"Expression: {row.expression}")
    print(f"Matched cue: {row.matched_cue}")
    print(f"Predicted label: {row.label}")

    while True:
        answer = input('Correct? [y/n]: ').strip().lower()
        if answer in {'y', 'yes', 'n', 'no'}:
            is_correct = answer.startswith('y')
            judgments.append(is_correct)
            break
        print('Please answer with y or n.')

    if is_correct:
        corrected_labels.append('')
        review_notes.append('')
    else:
        while True:
            corrected = input('Correct label [attribute/positional/relational]: ').strip().lower()
            if corrected in LABEL_ORDER:
                corrected_labels.append(corrected)
                break
            print(f"Please choose one of: {', '.join(LABEL_ORDER)}")
        review_notes.append(input('Notes (optional): ').strip())

sampled = sampled.copy()
sampled['human_correct'] = judgments
sampled['correct_label_if_wrong'] = corrected_labels
sampled['review_notes'] = review_notes
sampled[['expression', 'matched_cue', 'label', 'human_correct', 'correct_label_if_wrong', 'review_notes']].head()

In [ ]:
from pathlib import Path

# Precision is the share of predicted examples judged correct within each predicted label.
precision = sampled.groupby('label')['human_correct'].mean().reindex(LABEL_ORDER)
for label, value in precision.items():
    if pd.notna(value):
        print(f"{label}: {value:.3f}")

error_summary = (
    sampled.loc[~sampled['human_correct'], ['label', 'correct_label_if_wrong']]
    .value_counts()
    .rename('n')
    .reset_index()
)
if not error_summary.empty:
    display(error_summary)

output_path = Path('results/expression_classifier_audit.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
sampled.to_csv(output_path, index=False)
print(f"Saved audit annotations to {output_path}")

Paste the resulting round-4 per-label precision numbers into the README `Classifier Validation History` / `Manual Classifier Validation` section after finishing the audit.